In [ ]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 베이지안 최적화 라이브러리 설치
!pip install bayesian-optimization
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.2 MB/s eta 0:00:00


### XGB에 사용할 파라이터 예시
    random_state=42.
    max_depth=[3,4,5,6],         # 과적합 방지
    n_estimators=                # 트리 개수, 성능에 직결
    learning_rate=               # 학습 속도 조절 (n_estimators와 반비례하게 설정)

In [5]:
# 기본 라이브러리 할당
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd

# 데이터 분할 및 스케링 라이브러리 할당
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 모델 라이브러리 할당
from xgboost import XGBClassifier
import optuna

# 모델 평가 라이브러리 할당
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)
from sklearn.model_selection import cross_val_score

In [6]:
df_path = "/content/drive/MyDrive/OnSafe/modeling_final.csv"
df = pd.read_csv(df_path)

In [7]:
# 제외할 키워드 목록
exclude_keywords = ['video', 'file_id', 'frame', 'timestamp']

# 키워드가 하나라도 포함되면 제외
drop_col = [col for col in df.columns if not any(keyword in col.lower() for keyword in exclude_keywords)]

# 컬럼 선택
df = df[drop_col]

df

,neck_angle,neck_angular_velocity,neck_angular_acceleration,shoulder_balance_angle,shoulder_balance_angular_velocity,shoulder_balance_angular_acceleration,shoulder_left_angle,shoulder_left_angular_velocity,shoulder_left_angular_acceleration,shoulder_right_angle,...,spine_angular_acceleration,ankle_left_angle,ankle_left_angular_velocity,ankle_left_angular_acceleration,ankle_right_angle,ankle_right_angular_velocity,ankle_right_angular_acceleration,center_distance,center_speed,Label
0,20.680788,-12.271715,184.552498,117.980466,2498.058166,7052.299677,25.160310,365.426342,4574.281863,43.718651,...,-2055.415069,121.146172,-1041.416422,-5435.284408,134.870221,-1445.213398,-8822.845456,1.942890e-16,1.165734e-14,0.0
1,25.160022,6.151750,-3977.881772,126.860540,235.076656,-74874.669237,32.936111,152.476062,-12993.440283,49.397831,...,-6805.643321,107.223897,-181.176147,40327.731608,116.830968,-294.094849,53051.507406,3.955170e-16,2.373102e-14,0.0
2,20.885847,-144.867774,-425.943202,125.816355,2.235858,-8595.416016,30.242845,-67.688334,-5016.894535,50.575347,...,3083.921846,115.106967,302.841298,7890.131248,125.067060,323.170182,12170.580699,4.437638e-16,2.662583e-14,0.0
3,20.331096,-8.046357,4197.226968,126.935069,-51.437211,1911.131293,30.679833,-14.753756,-542.685302,53.972380,...,10712.091864,117.318607,81.828228,-9271.617768,127.603307,111.591175,-7018.747461,2.087443e-16,1.252466e-14,0.0
4,20.617635,-4.960209,-323.045662,124.101781,65.940234,8833.618161,29.751053,-85.777844,-941.782118,48.057296,...,-438.093245,117.834575,-6.212628,-5679.841082,128.786766,89.211933,-2096.623552,2.498002e-16,1.498801e-14,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296562,25.626388,31.437188,-739.170751,116.499138,-91.135295,1752.576522,14.011173,-2.501825,-726.467353,50.954784,...,-73.892263,116.095380,-24.295745,454.977069,107.489540,-32.471702,-799.993634,4.996004e-16,1.448841e-14,1.0
296563,26.102950,-14.325184,-1129.321421,114.687117,27.905029,3241.283093,13.426289,-28.977733,-756.436352,51.931975,...,11.409774,115.623805,14.651826,1599.422870,106.294996,-32.522526,696.040891,2.498002e-16,7.244205e-15,1.0
296564,24.638445,-46.447048,53.555766,118.423623,132.401470,-598.446070,12.012709,-54.669849,1400.294337,51.446722,...,-529.589874,117.105850,86.009281,375.720456,105.246607,15.531118,633.252007,3.532708e-16,1.024485e-14,1.0
296565,22.899705,-10.631683,7443.498688,123.818253,-13.367114,-12608.788308,9.655955,67.594290,6448.057128,58.201012,...,-9117.846468,121.555480,40.563581,-2798.553521,107.366108,11.150026,-79.421324,3.532708e-16,1.024485e-14,1.0


In [8]:
# X y
X = df.drop(columns='Label')
y = df['Label']

In [9]:
# Train/Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
# 정규화
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 다시 DataFrame으로 변환 (컬럼명 유지)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [11]:
# SMOTE : class 불균형 해소를 위한 합성 샘플 생성
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)  # ✅ y → y_train

print("원본 데이터 분포:\n", y.value_counts())
print("SMOTE 적용 후 데이터 분포:\n", y_train_res.value_counts())

원본 데이터 분포:
 Label
1.0    194917
0.0    101650
Name: count, dtype: int64
SMOTE 적용 후 데이터 분포:
 Label
1.0    155933
0.0    155933
Name: count, dtype: int64


In [12]:
def objective(trial):
    # 정수형 파라미터를 직접 지정할 수 있어 편리합니다.
    n_estimators = trial.suggest_int('n_estimators', 200, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.1)

    model = XGBClassifier(
        max_depth=3,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=42,
    )

    score = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='recall').mean()
    return score


# 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, callbacks=[optuna.study.MaxTrialsCallback(100, states=(optuna.trial.TrialState.COMPLETE,))])

print(f"최고 Recall 점수 : {study.best_value}")

[I 2026-05-16 16:33:38,194] A new study created in memory with name: no-name-4a2d09c6-0acb-4611-b46b-cf45abfbdf21
[I 2026-05-16 16:34:51,995] Trial 0 finished with value: 0.910121651500949 and parameters: {'n_estimators': 305, 'learning_rate': 0.08429946858017712}. Best is trial 0 with value: 0.910121651500949.
[I 2026-05-16 16:36:44,526] Trial 1 finished with value: 0.9141425794450351 and parameters: {'n_estimators': 485, 'learning_rate': 0.06423433783365606}. Best is trial 1 with value: 0.9141425794450351.
[I 2026-05-16 16:37:51,569] Trial 2 finished with value: 0.9078257690289491 and parameters: {'n_estimators': 272, 'learning_rate': 0.07564081207687845}. Best is trial 1 with value: 0.9141425794450351.
[I 2026-05-16 16:39:29,192] Trial 3 finished with value: 0.9110194584013559 and parameters: {'n_estimators': 415, 'learning_rate': 0.06390054496114589}. Best is trial 1 with value: 0.9141425794450351.
[I 2026-05-16 16:40:40,436] Trial 4 finished with value: 0.9098971994159865 and para

최고 Recall 점수 : 0.9220241803923834


In [13]:
# 가장 좋았던 파라미터로 최종 모델 생성
final_clf = XGBClassifier(**study.best_params)
final_clf.fit(X_train_res, y_train_res)

y_pred_dt = final_clf.predict(X_test_scaled)

print("Decision Tree 정확도:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

Decision Tree 정확도: 0.9330680783626125
Decision Tree Confusion Matrix:
 [[18244  2086]
 [ 1884 37100]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.90      0.90     20330
         1.0       0.95      0.95      0.95     38984

    accuracy                           0.93     59314
   macro avg       0.93      0.92      0.93     59314
weighted avg       0.93      0.93      0.93     59314



In [14]:
print(f"**study.best_params : {study.best_params}")

**study.best_params : {'n_estimators': 499, 'learning_rate': 0.09938257956319059}


In [15]:
def objective(trial):
    # 정수형 파라미터를 직접 지정할 수 있어 편리합니다.
    n_estimators = trial.suggest_int('n_estimators', 200, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.1)

    model = XGBClassifier(
        max_depth=4,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=42,
    )

    score = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='recall').mean()
    return score


# 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, callbacks=[optuna.study.MaxTrialsCallback(100, states=(optuna.trial.TrialState.COMPLETE,))])

print(f"최고 Recall 점수 : {study.best_value}")

[I 2026-05-16 17:21:30,030] A new study created in memory with name: no-name-e5e7b76d-13ee-4cd1-b0a5-12c862189f74
[I 2026-05-16 17:22:38,073] Trial 0 finished with value: 0.9258078560989356 and parameters: {'n_estimators': 211, 'learning_rate': 0.07762102817005057}. Best is trial 0 with value: 0.9258078560989356.
[I 2026-05-16 17:24:08,918] Trial 1 finished with value: 0.9274752269587225 and parameters: {'n_estimators': 315, 'learning_rate': 0.0686987657358807}. Best is trial 1 with value: 0.9274752269587225.
[I 2026-05-16 17:25:30,196] Trial 2 finished with value: 0.9260836303243801 and parameters: {'n_estimators': 278, 'learning_rate': 0.06589231115479861}. Best is trial 1 with value: 0.9274752269587225.
[I 2026-05-16 17:26:56,913] Trial 3 finished with value: 0.9278792439195589 and parameters: {'n_estimators': 297, 'learning_rate': 0.07477075278313604}. Best is trial 3 with value: 0.9278792439195589.
[I 2026-05-16 17:29:22,676] Trial 4 finished with value: 0.9321952126351857 and par

최고 Recall 점수 : 0.934131949110285


In [16]:
# 가장 좋았던 파라미터로 최종 모델 생성
final_clf = XGBClassifier(**study.best_params)
final_clf.fit(X_train_res, y_train_res)

y_pred_dt = final_clf.predict(X_test_scaled)

print("Decision Tree 정확도:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

Decision Tree 정확도: 0.9305223050207371
Decision Tree Confusion Matrix:
 [[18184  2146]
 [ 1975 37009]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

         0.0       0.90      0.89      0.90     20330
         1.0       0.95      0.95      0.95     38984

    accuracy                           0.93     59314
   macro avg       0.92      0.92      0.92     59314
weighted avg       0.93      0.93      0.93     59314



In [17]:
print(f"**study.best_params : {study.best_params}")

**study.best_params : {'n_estimators': 494, 'learning_rate': 0.08700909460632249}


### random_state도 설정 X

In [24]:
def objective(trial):
    # 정수형 파라미터를 직접 지정할 수 있어 편리합니다.
    n_estimators = trial.suggest_int('n_estimators', 200, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.1)
    random_state = trial.suggest_int('random_state', 0, 100)

    model = XGBClassifier(
        max_depth=3,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=random_state,
    )

    score = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='recall').mean()
    return score


# 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=70 , callbacks=[optuna.study.MaxTrialsCallback(100, states=(optuna.trial.TrialState.COMPLETE,))])

print(f"최고 Recall 점수 : {study.best_value}")

[I 2026-05-16 20:35:24,731] A new study created in memory with name: no-name-25e4d1b2-000b-4486-830e-e6bbe9dcf05a
[I 2026-05-16 20:36:50,241] Trial 0 finished with value: 0.911391411336778 and parameters: {'n_estimators': 286, 'learning_rate': 0.08627067231650426, 'random_state': 69}. Best is trial 0 with value: 0.911391411336778.
[I 2026-05-16 20:38:11,900] Trial 1 finished with value: 0.9116543488066478 and parameters: {'n_estimators': 338, 'learning_rate': 0.08317001122996007, 'random_state': 73}. Best is trial 1 with value: 0.9116543488066478.
[I 2026-05-16 20:39:31,791] Trial 2 finished with value: 0.9145337779517988 and parameters: {'n_estimators': 323, 'learning_rate': 0.09241120207278129, 'random_state': 78}. Best is trial 2 with value: 0.9145337779517988.
[I 2026-05-16 20:40:31,320] Trial 3 finished with value: 0.9076590528354711 and parameters: {'n_estimators': 226, 'learning_rate': 0.09338775288388049, 'random_state': 46}. Best is trial 2 with value: 0.9145337779517988.
[I 2

최고 Recall 점수 : 0.922607757705127


In [25]:
# 가장 좋았던 파라미터로 최종 모델 생성
final_clf = XGBClassifier(**study.best_params)
final_clf.fit(X_train_res, y_train_res)

y_pred_dt = final_clf.predict(X_test_scaled)

print("Decision Tree 정확도:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

Decision Tree 정확도: 0.932511717301143
Decision Tree Confusion Matrix:
 [[18241  2089]
 [ 1914 37070]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.90      0.90     20330
         1.0       0.95      0.95      0.95     38984

    accuracy                           0.93     59314
   macro avg       0.93      0.92      0.92     59314
weighted avg       0.93      0.93      0.93     59314



In [26]:
print(f"**study.best_params : {study.best_params}")

**study.best_params : {'n_estimators': 500, 'learning_rate': 0.09918821057323762, 'random_state': 99}


In [27]:
def objective(trial):
    # 정수형 파라미터를 직접 지정할 수 있어 편리합니다.
    n_estimators = trial.suggest_int('n_estimators', 200, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.1)
    random_state = trial.suggest_int('random_state', 0, 100)

    model = XGBClassifier(
        max_depth=4,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=random_state,
    )

    score = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='recall').mean()
    return score


# 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=80, callbacks=[optuna.study.MaxTrialsCallback(100, states=(optuna.trial.TrialState.COMPLETE,))])

print(f"최고 Recall 점수 : {study.best_value}")

[I 2026-05-16 22:32:32,020] A new study created in memory with name: no-name-af2cf48b-f506-41d5-ae48-0810e55803f2
[I 2026-05-16 22:34:08,664] Trial 0 finished with value: 0.9272507841273281 and parameters: {'n_estimators': 335, 'learning_rate': 0.0642635319839285, 'random_state': 76}. Best is trial 0 with value: 0.9272507841273281.
[I 2026-05-16 22:35:23,239] Trial 1 finished with value: 0.9198950679901919 and parameters: {'n_estimators': 238, 'learning_rate': 0.05299702806004402, 'random_state': 45}. Best is trial 0 with value: 0.9272507841273281.
[I 2026-05-16 22:37:12,116] Trial 2 finished with value: 0.9302713141494703 and parameters: {'n_estimators': 397, 'learning_rate': 0.06980879875900256, 'random_state': 61}. Best is trial 2 with value: 0.9302713141494703.
[I 2026-05-16 22:39:16,989] Trial 3 finished with value: 0.9308035991387857 and parameters: {'n_estimators': 454, 'learning_rate': 0.06435869483064664, 'random_state': 64}. Best is trial 3 with value: 0.9308035991387857.
[I 

최고 Recall 점수 : 0.9356197627026873


In [28]:
# 가장 좋았던 파라미터로 최종 모델 생성
final_clf = XGBClassifier(**study.best_params)
final_clf.fit(X_train_res, y_train_res)

y_pred_dt = final_clf.predict(X_test_scaled)

print("Decision Tree 정확도:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

Decision Tree 정확도: 0.933118656640928
Decision Tree Confusion Matrix:
 [[18249  2081]
 [ 1886 37098]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.90      0.90     20330
         1.0       0.95      0.95      0.95     38984

    accuracy                           0.93     59314
   macro avg       0.93      0.92      0.93     59314
weighted avg       0.93      0.93      0.93     59314



In [29]:
print(f"**study.best_params : {study.best_params}")

**study.best_params : {'n_estimators': 500, 'learning_rate': 0.09946909142793563, 'random_state': 7}
